# Model 1 vs OVATION

OVATION values: runs archived by [NASA CCMC](https://iswa.ccmc.gsfc.nasa.gov/iswa_data_tree/model/ionosphere/ovation_prime/e_data/),
built by `fetch_ovation_ccmc.py`. OVATION uses only real-time solar wind (not Kp or other indices directly); Model 1 is scored with final, cleaned data.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

bundle = joblib.load("aurora_ebm_model_final_deployment.joblib")
model1, featureslow = bundle["pipeline"], bundle["featureslow"]

cams = pd.read_csv("camera_minutes.csv.gz", parse_dates=["time"])
ovation = pd.read_csv("ovation_camera_minutes.csv.gz", parse_dates=["time"])
cams = cams.merge(ovation, on=["camera", "time"])
cams["abs_mlat"] = cams["mlat"].abs()
cams["oval_dist"] = cams["abs_mlat"] - (66.0 - 2.0 * cams["kp"])
cams["cloud_cover"] = 0.0
cams["model1"] = model1.predict_proba(cams[featureslow])[:, 1]

south = cams.copy()  # person 7° south of the camera, as in camera_calibration.ipynb
south["mlat"] = np.sign(south["mlat"]) * (south["abs_mlat"] - 7)
south["oval_dist"] = south["mlat"].abs() - (66.0 - 2.0 * south["kp"])
cams["model1_7_south"] = model1.predict_proba(south[featureslow])[:, 1]

order = cams.groupby("camera")["abs_mlat"].mean().sort_values().index
print(f"{len(cams):,} camera minutes with an OVATION value")

648,996 camera minutes with an OVATION value


## AUC within each camera (test 1)

"Kp only" is a baseline: minutes are simply ranked by their Kp value. At one fixed camera, Kp already carries most of the signal, so a model should at least match it. All three columns score the same clear, dark, moon-free camera minutes; with darkness, cloud and moon already removed and the location fixed, little besides Kp is left to vary.

In [2]:
auc = cams.groupby("camera").apply(lambda g: pd.Series({
    "Model 1": roc_auc_score(g["aurora"], g["model1"]),
    "OVATION": roc_auc_score(g["aurora"], g["ovation_eflux"]),
    "Kp only": roc_auc_score(g["aurora"], g["kp"])}), include_groups=False).loc[order]
print(auc.round(3).to_string())
print("\nmedian:", auc.median().round(2).to_dict())

             Model 1  OVATION  Kp only
camera                                
chibougamau    0.705    0.605    0.647
kapuskasing    0.735    0.604    0.733
athabasca      0.794    0.671    0.785
the pas        0.872    0.722    0.877
gakona         0.822    0.716    0.859
whitehorse     0.714    0.645    0.740
kiruna         0.801    0.783    0.825
skibotn        0.851    0.770    0.879
tromso         0.796    0.747    0.840

median: {'Model 1': 0.8, 'OVATION': 0.72, 'Kp only': 0.82}


## When Model 1 is confident (test 2)

Take the minutes where Model 1 says more than 70% and see how often the camera saw aurora. OVATION and Kp
each get the same number of their own highest-scoring minutes as Model 1 has above 70%. Note that they likely pick
different minutes; only the count is shared.

In [3]:
rng = np.random.default_rng(0)  # breaks ties: many minutes share the same OVATION or Kp value
rows = []
for c in order:
    g = cams[cams["camera"] == c]
    n = int((g["model1_7_south"] > 0.7).sum())
    if n == 0:
        continue
    top = lambda s: g.loc[(s + rng.uniform(0, 1e-9, len(g))).nlargest(n).index, "aurora"].mean()
    rows.append({"camera": c, "minutes": n, "Model 1": g.loc[g["model1_7_south"] > 0.7, "aurora"].mean(),
                 "OVATION": top(g["ovation_eflux"]), "Kp only": top(g["kp"])})
prec = pd.DataFrame(rows).set_index("camera")
print(prec.round(2).to_string())
print("\nmedian:", prec[["Model 1", "OVATION", "Kp only"]].median().round(2).to_dict())

             minutes  Model 1  OVATION  Kp only
camera                                         
kapuskasing      131     1.00     0.43     0.78
athabasca       2198     0.75     0.55     0.75
the pas         4836     0.77     0.58     0.71
gakona          2955     0.94     0.71     0.91
whitehorse      2833     0.91     0.71     0.93
kiruna         16040     0.93     0.93     0.95
skibotn        15254     0.97     0.92     0.97
tromso         64596     0.82     0.77     0.85

median: {'Model 1': 0.92, 'OVATION': 0.71, 'Kp only': 0.88}


Model 1 beats OVATION on both tests (median AUC 0.80 vs 0.72; 92% vs 71% when confident), but the gap could be
entirely due to its cleaner inputs. Against Kp only it is level: slightly behind on test 1 (0.80 vs 0.82), mixed camera
by camera on test 2 (92% vs 88% median; Kp ahead at three of eight cameras, Model 1 at three, two ties).